# Install Dependencies (5-7 minutes)

In [ ]:
!pip install nemo_toolkit['all'] wget typhoon-asr pydub openai
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU — go to Runtime > Change runtime type > T4 GPU")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


❌ No GPU — go to Runtime > Change runtime type > T4 GPU


# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/diarization_output"
SEGMENTS_DIR = "/content/segments"  # temp audio slices, NOT saved to Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SEGMENTS_DIR, exist_ok=True)
print(f"✅ Output will be saved to: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Output will be saved to: /content/drive/MyDrive/diarization_output


# Upload Audio

In [ ]:
from google.colab import files

print("Upload your audio file (wav, mp3, mp4, m4a)...")
uploaded = files.upload()

original_filename = list(uploaded.keys())[0]
audio_input_path = f"/content/{original_filename}"
print(f"✅ Uploaded: {audio_input_path}")

Upload your audio file (wav, mp3, mp4, m4a)...


Saving sound1.m4a to sound1.m4a
✅ Uploaded: /content/sound1.m4a


# Convert to 16kHz Mono WAV

In [ ]:
import subprocess

audio_wav_path = "/content/audio_input.wav"
!ffmpeg -y -i "{audio_input_path}" -ar 16000 -ac 1 "{audio_wav_path}"

result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", audio_wav_path],
    capture_output=True, text=True
)
duration = float(result.stdout.strip())
print(f"✅ Audio ready — Duration: {duration:.1f}s ({duration/60:.1f} min)")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# Create Manifest File

In [ ]:
import json

manifest_path = "/content/diar_manifest.json"

manifest_entry = {
    "audio_filepath": audio_wav_path,
    "offset": 0,
    "duration": None,
    "label": "infer",
    "text": "-",
    "num_speakers": None  # ← set e.g. 2 if you know the speaker count
}

with open(manifest_path, "w") as f:
    json.dump(manifest_entry, f)
    f.write("\n")

print(f"✅ Manifest created")

✅ Manifest created


# Run TitaNet Diarization

In [ ]:
import wget
from omegaconf import OmegaConf
from nemo.collections.asr.models import ClusteringDiarizer

config_path = "/content/diar_infer_meeting.yaml"
if not os.path.exists(config_path):
    wget.download(
        "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/"
        "speaker_tasks/diarization/conf/inference/diar_infer_meeting.yaml",
        config_path
    )

cfg = OmegaConf.load(config_path)

cfg.diarizer.manifest_filepath = manifest_path
cfg.diarizer.out_dir           = OUTPUT_DIR

cfg.diarizer.vad.model_path                      = "vad_multilingual_marblenet"
cfg.diarizer.vad.parameters.onset                = 0.8
cfg.diarizer.vad.parameters.offset               = 0.6
cfg.diarizer.vad.parameters.min_duration_on      = 0.1
cfg.diarizer.vad.parameters.min_duration_off     = 0.4

cfg.diarizer.speaker_embeddings.model_path                      = "titanet_large"
cfg.diarizer.speaker_embeddings.parameters.window_length_in_sec = [1.5, 1.25, 1.0, 0.75, 0.5]
cfg.diarizer.speaker_embeddings.parameters.shift_length_in_sec  = [0.75, 0.625, 0.5, 0.375, 0.25]
cfg.diarizer.speaker_embeddings.parameters.multiscale_weights    = [1, 1, 1, 1, 1]

cfg.diarizer.clustering.parameters.oracle_num_speakers = False
cfg.diarizer.clustering.parameters.max_num_speakers    = 8

print("🚀 Running TitaNet diarization...")
diarizer = ClusteringDiarizer(cfg=cfg)
diarizer.diarize()
print("✅ Diarization complete!")

[NeMo W 2026-04-10 02:25:15 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-04-10 02:25:15 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-04-10 02:25:15 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-04-10 02:25:15 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


🚀 Running TitaNet diarization...
[NeMo I 2026-04-10 02:25:16 clustering_diarizer:117] Loading pretrained vad_multilingual_marblenet model from NGC
[NeMo I 2026-04-10 02:25:16 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/vad_multilingual_marblenet/versions/1.10.0/files/vad_multilingual_marblenet.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo
[NeMo I 2026-04-10 02:25:17 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-04-10 02:25:17 classification_models:641] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
[NeMo W 2026-04-10 02:25:17 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/ami_train_0.63.json,/manifests/freesound_background_train.json,/manifests/freesound_laughter_train.json,/manifests/fisher_2004_background.json,/manifests/fisher_2004_speech_sampled.json,/manifests/google_train_manifest.json,/manifests/icsi_all_0.63.json,/manifests/musan_freesound_train.json,/manifests/musan_music_train.json,/manifests/musan_soundbible_train.json,/manifests/mandarin_train_sample.json,/manifests/german_train_sample.json,/manifests/spanish_train_sample.json,/manifests/french_train_sample.json,/manifests/russian_tr

[NeMo I 2026-04-10 02:25:17 save_restore_connector:285] Model EncDecClassificationModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2026-04-10 02:25:17 clustering_diarizer:150] Loading pretrained titanet_large model from NGC
[NeMo I 2026-04-10 02:25:17 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/v1/files/titanet-l.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2026-04-10 02:25:18 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-04-10 02:25:20 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-04-10 02:25:20 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method 

...

[NeMo I 2026-04-10 03:23:36 clustering_diarizer:383] Saved embedding files to /content/drive/MyDrive/diarization_output/speaker_outputs/embeddings


[NeMo W 2026-04-10 03:23:36 speaker_utils:473] cuda=False, using CPU for eigen decomposition. This might slow down the clustering process.
clustering: 100%|██████████| 1/1 [00:05<00:00,  5.72s/it]

[NeMo I 2026-04-10 03:23:42 clustering_diarizer:451] Outputs are saved in /content/drive/MyDrive/diarization_output directory



[NeMo W 2026-04-10 03:23:42 der:221] Check if each ground truth RTTMs were present in the provided manifest file. Skipping calculation of Diariazation Error Rate


✅ Diarization complete!


# Parse RTTM File

**RTTM (Rich Transcription Time Marked)** is a space-delimited text format used primarily for Speaker Diarization. It acts as a map of "who spoke when" within an audio recording.

Each line in the file represents a single speech segment and typically includes:

**Start Time:** When the speaker began talking (in seconds).

**Duration:** How long the segment lasted (in seconds).

**Speaker ID:** A unique label for the person speaking (e.g., SPEAKER_01).

In [ ]:
import glob

rttm_files = glob.glob(os.path.join(OUTPUT_DIR, "pred_rttms", "*.rttm"))
assert rttm_files, "❌ No RTTM file found — check Cell 7 output for errors"

rttm_path = rttm_files[0]
print(f"📄 RTTM: {rttm_path}\n")

segments = []
with open(rttm_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        start    = float(parts[3])
        duration = float(parts[4])
        speaker  = parts[7]
        segments.append({
            "start":    start,
            "end":      start + duration,
            "duration": duration,
            "speaker":  speaker
        })

print(f"✅ Found {len(segments)} segments, "
      f"{len(set(s['speaker'] for s in segments))} speakers")

📄 RTTM: /content/drive/MyDrive/diarization_output/pred_rttms/audio_input.rttm

✅ Found 20 segments, 4 speakers


# Slice Audio Per Segment

In [ ]:
from pydub import AudioSegment

print("✂️  Slicing audio into speaker segments...")
audio = AudioSegment.from_wav(audio_wav_path)

for i, seg in enumerate(segments):
    start_ms = int(seg["start"] * 1000)
    end_ms   = int(seg["end"]   * 1000)
    clip     = audio[start_ms:end_ms]
    out_path = os.path.join(SEGMENTS_DIR, f"seg_{i:04d}.wav")
    clip.export(out_path, format="wav")
    seg["clip_path"] = out_path  # store path back into segment dict

print(f"✅ {len(segments)} audio slices saved to {SEGMENTS_DIR}")

✂️  Slicing audio into speaker segments...
✅ 20 audio slices saved to /content/segments


# Load model ASR + Transcript

In [ ]:
import torch
import nemo.collections.asr as nemo_asr
from huggingface_hub import snapshot_download
import os

# ── Load model ONCE ─────────────────────────────────────────
print("🌪️ Loading Typhoon ASR model (once)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Download model from HuggingFace if not cached
model_dir = snapshot_download("scb10x/typhoon-asr-realtime")
nemo_file = os.path.join(model_dir, "typhoon-asr-realtime.nemo")

asr_model = nemo_asr.models.EncDecRNNTBPEModel.restore_from(nemo_file)
asr_model = asr_model.to(device)
asr_model.eval()
print(f"✅ Model loaded on {device.upper()}\n")

# ── Transcribe all segments ──────────────────────────────────
print(f"🎙️ Transcribing {len(segments)} segments...")

# Collect valid segment paths
valid_segments = [(i, seg) for i, seg in enumerate(segments)
                  if seg["duration"] >= 0.3]
short_segments = [(i, seg) for i, seg in enumerate(segments)
                  if seg["duration"] < 0.3]

# Mark short ones immediately
for i, seg in short_segments:
    seg["text"] = "[too short]"

# Batch transcribe all valid segments at once
clip_paths = [seg["clip_path"] for _, seg in valid_segments]

if clip_paths:
    # transcribe() returns a list of strings directly when called on the model
    with torch.no_grad():
        transcriptions = asr_model.transcribe(clip_paths, batch_size=8)

    # transcriptions may be a list of strings or list of Hypothesis — handle both
    for (i, seg), result in zip(valid_segments, transcriptions):
        if isinstance(result, str):
            text = result
        elif hasattr(result, "text"):
            text = result.text
        elif hasattr(result, "y_sequence"):
            # Some NeMo versions store text differently
            text = str(result)
        else:
            text = str(result)
        seg["text"] = text.strip() if text.strip() else "[silence]"

print(f"✅ Transcription complete! ({len(valid_segments)} segments transcribed)")

🌪️ Loading Typhoon ASR model (once)...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[NeMo I 2026-04-10 03:45:34 mixins:184] Tokenizer SentencePieceTokenizer initialized with 2048 tokens


...

[NeMo I 2026-04-10 03:45:36 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--scb10x--typhoon-asr-realtime/snapshots/1aaf2ea81762b311896898f4c4a376d227865076/typhoon-asr-realtime.nemo.


[NeMo W 2026-04-10 03:45:36 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-10 03:45:36 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


✅ Model loaded on CPU

🎙️ Transcribing 20 segments...


Transcribing: 3it [00:38, 12.94s/it]

✅ Transcription complete! (19 segments transcribed)


# Print final Transcript

In [ ]:
print("=" * 60)
print("📝  SPEAKER-ATTRIBUTED TRANSCRIPT")
print("=" * 60 + "\n")

for seg in segments:
    m_s, s_s = divmod(int(seg["start"]), 60)
    m_e, s_e = divmod(int(seg["end"]),   60)
    timestamp = f"[{m_s:02d}:{s_s:02d} → {m_e:02d}:{s_e:02d}]"
    print(f"{timestamp}  {seg['speaker']}")
    print(f"  {seg['text']}\n")

📝  SPEAKER-ATTRIBUTED TRANSCRIPT

[00:00 → 00:12]  speaker_2
  สรรปกครองสรรปกครองก็แค่หลับเรื่องไว้นะฮะต้องรอดูว่าจะพิจารณาไว้พิจารณายังไงด้วยนะศาลรัฐธรรมนูญที่จะสามารถสั่งได้โมคะหรือเลือกตั้งใหม่ยังไงก็ต้องรอดูว่าจะมีใครไปยื่นยังไงหรือเปล่าแต่เขตหนึ่งชลบุรีนี่เขาเป็นยังไงนะ

[00:12 → 00:37]  speaker_0
  โอ้โห เขตหนึ่งชลบุรีเนี่ย ต้องบอกว่าเรื่องราวดราม่ากันไม่หยุดเลยนะครับ คือประเด็นที่คนเขาไปทวงถามกันเลยก็คือกล้องวงจรปิดอะ ที่สนามแบตมินตันนะครับ แล้วมันเป็นสถานที่ในการเอาหีบการเลือกตั้งแต่ละหน่วย เอามารวมกันไว้ ปรากฏว่าไฟหายอยู่วันเดียวคุณผู้ชมครับ วันที่เก้ากุมภาพันธ์เวลากลางคืน ซึ่งมันเป็นเหตุการณ์ที่แต่ละหน่วยเนี่ยเขากําลังเอาหีบมารวมกันที่นี่

[00:37 → 00:38]  speaker_2
  คือวันเกิดเหตุ

[00:38 → 00:39]  speaker_0
  เกิดเหตุเลย

[00:40 → 00:41]  speaker_0
  นะฮะ

[00:41 → 01:05]  speaker_0
  คนที่ออกมาเคลื่อนไหวเนี่ยอย่างทางด้านเฟิร์นกระหนกวันนะครับก็เป็นตัวแทนกลุ่มประชาชนก็บอกว่าวันนี้เนี่ยทั้งทีมผู้เชี่ยวชาญของโครงการอินเทอร์เน็ตนะครับตามกฎหมายเพื่อประชาชนหรือว่าไอลอก็จะไปประสา